In [12]:
import cudf 
import cudf as df
import cupy as cp
import numpy as np
import gc
import os
import time

from cuml.preprocessing import StandardScaler
from cuml.neighbors import NearestNeighbors

print("cuDF version :", cudf.__version__)
print("CuPy version :", cp.__version__)

try:
    import cuml
    print("cuML version :", cuml.__version__)
except Exception:
    print("cuML loaded")


cuDF version : 26.06.01
CuPy version : 14.2.0
cuML version : 26.06.00


In [4]:

print("=" * 70)
print("GPU CHECK")
print("=" * 70)

try:
    device = cp.cuda.Device()
    print("GPU ID:", device.id)

    free_mem, total_mem = cp.cuda.runtime.memGetInfo()

    print(
        f"GPU total memory : "
        f"{total_mem / (1024**3):.2f} GB"
    )

    print(
        f"GPU free memory  : "
        f"{free_mem / (1024**3):.2f} GB"
    )

except Exception as e:
    print("GPU check failed:")
    print(e)


GPU CHECK
GPU ID: 0
GPU total memory : 6.00 GB
GPU free memory  : 4.95 GB


In [29]:
start = time.time()

train_transaction = cudf.read_csv(
    r"/home/abhin/mainproject/FEDBANK/data/train_transaction.csv"
)
train_identity=cudf.read_csv(r"/home/abhin/mainproject/FEDBANK/data/train_identity.csv")
print("Transaction shape:", train_transaction.shape)
print("Time:", round(time.time() - start, 2), "seconds")

Transaction shape: (590540, 394)
Time: 31.71 seconds


In [30]:
start = time.time()

df = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print("Merged shape:", df.shape)
print("Time:", round(time.time() - start, 2), "seconds")

Merged shape: (590540, 434)
Time: 0.41 seconds


In [31]:
print("Number of rows   :", len(df))
print("Number of columns:", len(df.columns))

print("\nFirst 20 columns:")
print(df.columns[:20])

print("\nLast 20 columns:")
print(df.columns[-20:])

Number of rows   : 590540
Number of columns: 434

First 20 columns:
Index(['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
       'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain',
       'C1', 'C2', 'C3'],
      dtype='object')

Last 20 columns:
Index(['id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28',
       'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36',
       'id_37', 'id_38', 'DeviceType', 'DeviceInfo'],
      dtype='object')


In [32]:
print(df["isFraud"].value_counts())

fraud_count = int((df["isFraud"] == 1).sum())
normal_count = int((df["isFraud"] == 0).sum())

print("\nNormal transactions:", normal_count)
print("Fraud transactions :", fraud_count)

print(
    "Fraud percentage:",
    round((fraud_count / len(df)) * 100, 4),
    "%"
)

isFraud
0    569877
1     20663
Name: count, dtype: int64

Normal transactions: 569877
Fraud transactions : 20663
Fraud percentage: 3.499 %


In [33]:
null_report = cudf.DataFrame({
    "column": df.columns,
    "missing_count": [
        int(df[col].isna().sum())
        for col in df.columns
    ]
})

null_report["missing_percent"] = (
    null_report["missing_count"] / len(df) * 100
)

null_report = null_report.sort_values(
    "missing_percent",
    ascending=False
)

null_report.head(30)

,column,missing_count,missing_percent
417,id_24,585793,99.196159
418,id_25,585408,99.130965
400,id_07,585385,99.127070
401,id_08,585385,99.127070
414,id_21,585381,99.126393
419,id_26,585377,99.125715
415,id_22,585371,99.124699
416,id_23,585371,99.124699
420,id_27,585371,99.124699
14,dist2,552913,93.628374


In [34]:
v_features = [
    col for col in df.columns
    if col.startswith("V")
]

print("Number of V features:", len(v_features))

print(v_features[:20])

Number of V features: 339
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20']


In [35]:
v_report = cudf.DataFrame({
    "feature": v_features,
    "missing_count": [
        int(df[col].isna().sum())
        for col in v_features
    ]
})

v_report["missing_percent"] = (
    v_report["missing_count"] / len(df) * 100
)

v_report = v_report.sort_values(
    "missing_percent",
    ascending=False
)

v_report.head(30)

,feature,missing_count,missing_percent
137,V138,508595,86.123717
138,V139,508595,86.123717
139,V140,508595,86.123717
140,V141,508595,86.123717
141,V142,508595,86.123717
145,V146,508595,86.123717
146,V147,508595,86.123717
147,V148,508595,86.123717
148,V149,508595,86.123717
152,V153,508595,86.123717


In [ ]:
# ============================================================
# FEATURE CATEGORIZATION
# ============================================================

very_high_missing = []
high_missing = []
knn_features = []
complete_features = []

for col in v_features:

    missing_percent = (
        df[col].isna().sum() / len(df) * 100
    )

    if missing_percent >= 90:
        very_high_missing.append(col)

    elif missing_percent >= 50:
        high_missing.append(col)

    elif missing_percent > 0:
        knn_features.append(col)

    else:
        complete_features.append(col)

print("Feature categorization")
print("=" * 50)
print(">= 90% missing :", len(very_high_missing))
print("50-90% missing :", len(high_missing))
print("< 50% missing  :", len(knn_features))
print("No missing     :", len(complete_features))
print("Total V        :", len(v_features))

High-missing features : 159
KNN features          : 180
Complete features     : 0


In [40]:
missing_indicator_features = []

for col in v_features:

    missing_percent = (
        df[col].isna().sum() / len(df) * 100
    )

    if missing_percent > 5:

        indicator_name = col + "_missing"

        df[indicator_name] = df[col].isna().astype("int8")

        missing_indicator_features.append(indicator_name)

print(
    "Created",
    len(missing_indicator_features),
    "missing indicators."
)

Created 253 missing indicators.


In [41]:
numeric_columns = df.select_dtypes(
    include=["float32", "float64", "int8", "int16", "int32", "int64"]
).columns

for col in numeric_columns:

    df[col] = df[col].replace(
        [cp.inf, -cp.inf],
        None
    )

print("Infinite-value cleanup completed.")

Infinite-value cleanup completed.


In [42]:
start = time.time()

median_imputed_features = (
    very_high_missing + high_missing
)

for col in median_imputed_features:

    median_value = df[col].median()

    if median_value is not None:
        df[col] = df[col].fillna(median_value)

print(
    "Median imputation completed for",
    len(median_imputed_features),
    "features."
)

print("Time:", round(time.time() - start, 2), "seconds")

Median imputation completed for 159 features.
Time: 1.3 seconds


In [43]:
remaining_report = cudf.DataFrame({
    "feature": v_features,
    "missing_count": [
        int(df[col].isna().sum())
        for col in v_features
    ]
})

remaining_report["missing_percent"] = (
    remaining_report["missing_count"] / len(df) * 100
)

remaining_report = remaining_report.sort_values(
    "missing_percent",
    ascending=False
)

remaining_report.head(30)

,feature,missing_count,missing_percent
0,V1,279287,47.293494
1,V2,279287,47.293494
2,V3,279287,47.293494
3,V4,279287,47.293494
4,V5,279287,47.293494
5,V6,279287,47.293494
6,V7,279287,47.293494
7,V8,279287,47.293494
8,V9,279287,47.293494
9,V10,279287,47.293494


In [44]:
knn_feature_mask = df[knn_features].notna().all(axis=1)

complete_indices = df.index[knn_feature_mask]

print(
    "Complete reference rows:",
    len(complete_indices)
)

Complete reference rows: 254676


In [45]:
REFERENCE_SIZE = 20_000

if len(complete_indices) > REFERENCE_SIZE:

    reference_indices = complete_indices.to_series().sample(
        n=REFERENCE_SIZE,
        random_state=42
    ).index

else:

    reference_indices = complete_indices

print(
    "Reference rows selected:",
    len(reference_indices)
)

Reference rows selected: 20000


In [46]:
reference_df = df.loc[
    reference_indices,
    knn_features
].astype("float32")

print("Reference shape:", reference_df.shape)

Reference shape: (20000, 180)


In [47]:
reference_gpu = cp.asarray(
    reference_df.to_cupy()
)

print("Reference GPU shape:", reference_gpu.shape)
print("GPU dtype:", reference_gpu.dtype)

Reference GPU shape: (20000, 180)
GPU dtype: float32


In [48]:
reference_mean = cp.nanmean(
    reference_gpu,
    axis=0
)

print(
    "Reference mean shape:",
    reference_mean.shape
)

Reference mean shape: (180,)


In [49]:
reference_filled = cp.where(
    cp.isnan(reference_gpu),
    reference_mean,
    reference_gpu
)

print(
    "NaNs remaining in reference:",
    int(cp.isnan(reference_filled).sum())
)

NaNs remaining in reference: 0


In [50]:
scaler = StandardScaler()

reference_scaled = scaler.fit_transform(
    reference_filled
)

print(
    "Scaled reference shape:",
    reference_scaled.shape
)

Scaled reference shape: (20000, 180)


In [51]:
K = 5

knn = NearestNeighbors(
    n_neighbors=K
)

knn.fit(reference_scaled)

print("GPU KNN model ready.")

GPU KNN model ready.


In [52]:
missing_knn_mask = df[knn_features].isna().any(axis=1)

missing_knn_indices = df.index[missing_knn_mask]

print(
    "Rows requiring KNN imputation:",
    len(missing_knn_indices)
)

Rows requiring KNN imputation: 335864


In [53]:
def gpu_knn_impute_batch(
    batch_df,
    reference_gpu,
    reference_scaled,
    reference_mean,
    knn,
    scaler,
    k
):

    # Original batch values
    batch_gpu = cp.asarray(
        batch_df.to_cupy()
    ).astype(cp.float32)

    # Missing-value mask
    missing_mask = cp.isnan(batch_gpu)

    # Fill missing values temporarily for neighbor search
    batch_filled = cp.where(
        missing_mask,
        reference_mean,
        batch_gpu
    )

    # Scale batch
    batch_scaled = scaler.transform(
        batch_filled
    )

    # Find nearest neighbours
    distances, indices = knn.kneighbors(
        batch_scaled,
        n_neighbors=k
    )

    # Get original reference values
    neighbor_values = reference_gpu[
        indices
    ]

    # Mean value for each feature across neighbours
    neighbor_means = cp.mean(
        neighbor_values,
        axis=1
    )

    # Replace only missing values
    result = cp.where(
        missing_mask,
        neighbor_means,
        batch_gpu
    )

    return result

In [ ]:
BATCH_SIZE = 1000

start = time.time()

total_.
rows = len(missing_knn_indices)

print(
    "Rows to process:",
    total_rows
)

for start_pos in range(
    0,
    total_rows,
    BATCH_SIZE
):

    end_pos = min(
        start_pos + BATCH_SIZE,
        total_rows
    )

    batch_indices = missing_knn_indices[
        start_pos:end_pos
    ]

    batch_df = df.loc[
        batch_indices,
        knn_features
    ].astype("float32")

    batch_result = gpu_knn_impute_batch(
        batch_df=batch_df,
        reference_gpu=reference_gpu,
        reference_scaled=reference_scaled,
        reference_mean=reference_mean,
        knn=knn,
        scaler=scaler,
        k=K
    )

    result_df = cudf.DataFrame(
        batch_result,
        columns=knn_features,
        index=batch_indices
    )

    df.loc[
        batch_indices,
        knn_features
    ] = result_df

    if (
        start_pos % (BATCH_SIZE * 10) == 0
        or end_pos == total_rows
    ):

        percent = (
            end_pos / total_rows
        ) * 100

        print(
            f"Progress: {end_pos}/{total_rows} "
            f"({percent:.2f}%)"
        )

    del batch_df
    del batch_result
    del result_df

    cp.get_default_memory_pool().free_all_blocks()
    gc.collect()

print(
    "\nKNN imputation completed."
)

print(
    "Time:",
    round(time.time() - start, 2),
    "seconds"
)

Rows to process: 335864


ValueError: shape mismatch: value array of shape (1000, 180) could not be broadcast to indexing result of shape (1000, 687)